# 3 · Embeddings + pgvector + Retrieval-Augmented Generation | 50 minutes
Build a complete RAG loop: **documents → embeddings → vector store → similarity search → OpenAI answer grounded in sources**.

The instructor already embedded 18 fictional KB articles into Supabase (`shared.knowledge_articles`). You will create **one new article**, store its OpenAI embedding in your team table, and ask grounded questions. **The Kaggle `Resolution` text is not used as the knowledge base**—it is synthetic/noisy and not a trustworthy support policy.

**OpenAI credentials:** use your own *budgeted* API key or instructor-provided authenticated proxy. Never use/distribute the instructor's master API key. Embeddings cost API usage.

In [ ]:
%pip install -q "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "plotly>=5,<7" "openai>=1.55,<3"

In [ ]:
import os, getpass
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

In [ ]:
import os, getpass
from openai import OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass.getpass("Your budgeted OpenAI API key (never instructor master key): ")
client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"      # 1536 dimensions by default
CHAT_MODEL = "gpt-4.1-mini"

def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

def vector_literal(vector):
    assert len(vector) == 1536, "Embedding dimension must match vector(1536)"
    return "[" + ",".join(str(float(x)) for x in vector) + "]"

## Explore instructor-prepared KB

In [ ]:
display(fetch_df("SELECT article_id,title,category,LEFT(content,100) AS preview FROM shared.knowledge_articles ORDER BY article_id LIMIT 8"))
print("Vector-enabled KB rows:", int(fetch_df("SELECT COUNT(*) AS n FROM shared.knowledge_articles").iloc[0,0]))

## TODO 1 · Create your own embedding

In [ ]:
new_id=f"TEAM-{TEAM_NUMBER:02d}-001"
article_title="Troubleshooting Bluetooth pairing"
article_content="Check Bluetooth is enabled on both devices, remove stale pairings, restart pairing mode, and verify device compatibility. Escalate if repeated attempts fail."
# TODO 1: Call embed with title + content, not an empty string
new_embedding = embed("TODO: replace with article_title + article_content")
print("Embedding dimensions:", len(new_embedding))
assert len(new_embedding)==1536

## Store embedding in your own schema (provided)

In [ ]:
with closing(connect()) as conn:
    with conn.transaction():
        with conn.cursor() as cur:
            cur.execute(f"""INSERT INTO {TEAM_SCHEMA}.knowledge_articles
                (article_id,title,content,embedding) VALUES (%s,%s,%s,%s::extensions.vector)
                ON CONFLICT (article_id) DO UPDATE SET title=EXCLUDED.title,
                content=EXCLUDED.content,embedding=EXCLUDED.embedding""",
                (new_id,article_title,article_content,vector_literal(new_embedding)))
print("Saved:",new_id)

## TODO 2 · Similarity search in PostgreSQL

In [ ]:
question="How do I fix Bluetooth pairing problems?"
question_embedding = vector_literal(embed(question))
# TODO 2: Fill in the cosine-distance operator: <=>
query=f"""SELECT article_id,title,content,
               1 - (embedding TODO_OPERATOR %s::extensions.vector) AS similarity
          FROM (SELECT article_id,title,content,embedding FROM shared.knowledge_articles
                UNION ALL
                SELECT article_id,title,content,embedding FROM {TEAM_SCHEMA}.knowledge_articles) kb
          ORDER BY embedding TODO_OPERATOR %s::extensions.vector LIMIT 3"""
passages=fetch_df(query,(question_embedding,question_embedding))
display(passages[["article_id","title","similarity"]])

## TODO 3 · Build a grounded prompt; generate an OpenAI answer

In [ ]:
context="\n\n".join(f"[{r.article_id}] {r.title}: {r.content}" for r in passages.itertuples())
# TODO 3: Provide the retrieved context; ask the LLM to cite article IDs.
system_prompt = "Answer ONLY from knowledge articles. Cite article IDs. If unsure, say I do not know."
user_prompt = f"Question: {question}\n\nKnowledge articles:\n{ "TODO: replace with context" }"
response=client.responses.create(model=CHAT_MODEL,input=[
    {"role":"system","content":system_prompt},
    {"role":"user","content":user_prompt}
])
print(response.output_text)

In [ ]:
def retrieve_articles(question, k=3):
    q_vec = vector_literal(embed(question))
    # Instructor-provisioned views/tables; schema is constructed only from a numeric team ID.
    query = f"""
      SELECT article_id, title, content, 1 - (embedding <=> %s::extensions.vector) AS similarity
      FROM (
         SELECT article_id, title, content, embedding FROM shared.knowledge_articles
         UNION ALL
         SELECT article_id, title, content, embedding FROM {TEAM_SCHEMA}.knowledge_articles
      ) kb
      WHERE embedding IS NOT NULL
      ORDER BY embedding <=> %s::extensions.vector
      LIMIT %s
    """
    return fetch_df(query, (q_vec,q_vec,int(k)))

def grounded_answer(question, passages):
    context = "\n\n".join(
        f"[{row.article_id}] {row.title}: {row.content}" for row in passages.itertuples()
    )
    response = client.responses.create(
        model=CHAT_MODEL,
        input=[
          {"role":"system","content": "Answer only from retrieved knowledge articles. Cite the article IDs in square brackets. If context does not answer the question, say you do not know. Do not invent policies."},
          {"role":"user","content":f"Question: {question}\n\nRetrieved articles:\n{context}"},
        ],
    )
    return response.output_text

## Test an unsupported question

In [ ]:
# Try another question, and discuss why RAG cannot guarantee every answer is factual.
question2="What is the guaranteed refund amount for a damaged device?"
passages2=retrieve_articles(question2,3)
print(grounded_answer(question2,passages2))

### RAG checkpoint
1. Verify your team article is retrieved for the Bluetooth question.
2. Inspect whether source citations actually match article IDs.
3. Explain why the LLM should refuse to invent refund policies.
4. Contrast `SQL COUNT(*)` with semantic search: one computes facts from rows; the other finds relevant prose.